In [2]:
import json
import math
import sys
from collections import Counter

In [3]:
def safe_float(value, default=None):
    """Convert value to float safely. Returns default (None) if conversion fails.""" 
    if value is None: 
        return default 
        try: 
            return float(value) 
        except (ValueError, TypeError): 
            return default

Load and Inspect Top-Level Structure

In [4]:
def load_cityjson(filepath):
    """Load a CityJSON file and return the parsed JSON."""
    print(f"\n{'='*70}")
    print(f"Loading: {filepath}")
    print(f"{'='*70}")

    try:
        with open(filepath, 'r') as f:
            cj = json.load(f)
    except FileNotFoundError:
        print(f"ERROR: File not found: {filepath}")
        sys.exit(1)
    except json.JSONDecodeError as e:
        print(f"ERROR: Invalid JSON in {filepath}: {e}")
        sys.exit(1)

    print(f"\nCityJSON version: {cj.get('version', 'unknown')}")
    print(f"Top-level keys: {list(cj.keys())}")

    if 'CityObjects' not in cj:
        print("WARNING: No 'CityObjects' key found — may not be valid CityJSON")
    if 'vertices' not in cj:
        print("WARNING: No 'vertices' key found — geometry cannot be resolved")

    transform = cj.get('transform', {})
    scale = transform.get('scale', [1, 1, 1])
    translate = transform.get('translate', [0, 0, 0])
    print(f"\nTransform:")
    print(f"  Scale:     {scale}")
    print(f"  Translate: {translate}")
    if not transform:
        print("  WARNING: No transform found — vertices may already be real coords")
    print(f"  → Real coordinate = (integer_vertex * scale) + translate")

    vertices = cj.get('vertices', [])
    print(f"\nTotal shared vertices: {len(vertices)}")

    city_objects = cj.get('CityObjects', {})
    print(f"Total CityObjects: {len(city_objects)}")

    metadata = cj.get('metadata', {})
    if metadata:
        print(f"Reference system: {metadata.get('referenceSystem', 'unknown')}")

    return cj

In [5]:
def show_transform_example(cj):
    """Demonstrate how CityJSON vertex transformation works."""
    print(f"\n{'='*70}")
    print("VERTEX TRANSFORMATION")
    print(f"{'='*70}")

    transform = cj.get('transform', {})
    scale = transform.get('scale', [1, 1, 1])
    translate = transform.get('translate', [0, 0, 0])
    vertices = cj.get('vertices', [])

    if not vertices:
        print("  No vertices found — skipping transform examples.")
        return scale, translate

    print(f"""
CityJSON stores vertices as INTEGERS to save space.
Real-world coords: real = integer * scale + translate

Scale:     {scale}
Translate: {translate}
""")

    num_examples = min(6, len(vertices))
    print(f"{'Index':<8} {'Integer (x,y,z)':<25} {'Real-world (x,y,z) [meters]'}")
    print("-" * 70)
    for i in range(num_examples):
        v = vertices[i]
        if not isinstance(v, (list, tuple)) or len(v) < 3:
            print(f"{i:<8} {str(v):<25} (INVALID VERTEX FORMAT)")
            continue
        real = [
            v[0] * scale[0] + translate[0],
            v[1] * scale[1] + translate[1],
            v[2] * scale[2] + translate[2]
        ]
        print(f"{i:<8} {str(v):<25} ({real[0]:.3f}, {real[1]:.3f}, {real[2]:.3f})")

    return scale, translate

CityObject Hierarchy

In [ ]:
'''
def explore_city_objects(cj):
    """Analyze the CityObject types and their hierarchy."""
    print(f"\n{'='*70}")
    print("CITYOBJECT HIERARCHY")
    print(f"{'='*70}")

    city_objects = cj.get('CityObjects', {})
    if not city_objects:
        print("  No CityObjects found.")
        return {}

    type_counts = Counter()
    for obj_id, obj in city_objects.items():
        type_counts[obj.get('type', 'unknown')] += 1

    print(f"\nObject type distribution:")
    for t, count in type_counts.most_common():
        print(f"  {t}: {count}")

    print(f"""
HIERARCHY:
  Building (parent) → contains attributes, no geometry
  BuildingPart (child) → contains geometry at multiple LODs
""")

    buildings = {k: v for k, v in city_objects.items()
                 if v.get('type') == 'Building'}

    max_show = min(20, len(buildings))
    shown = 0
    warnings = {'no_children': 0, 'missing_ref': 0, 'missing_lods': 0}

    for bld_id, bld in buildings.items():
        if shown >= max_show:
            remaining = len(buildings) - max_show
            if remaining > 0:
                print(f"\n  ... and {remaining} more buildings")
            break

        children = bld.get('children', [])
        attrs = bld.get('attributes', {})
        dak_type = attrs.get('b3_dak_type', 'N/A') if attrs else 'N/A'
        bouwjaar = attrs.get('oorspronkelijk_bouwjaar', 'N/A') if attrs else 'N/A'

        if not children:
            warnings['no_children'] += 1

        print(f"\n  Building: {bld_id}")
        print(f"    Roof type (b3_dak_type): {dak_type}")
        print(f"    Construction year: {bouwjaar}")
        print(f"    Children: {len(children)}")

        for child_id in children:
            child = city_objects.get(child_id)
            if child is None:
                warnings['missing_ref'] += 1
                print(f"      └─ {child_id} (WARNING: not found!)")
                continue
            geom_list = child.get('geometry', [])
            lods = [g.get('lod', '?') for g in geom_list]
            has_both = '1.3' in lods and '2.2' in lods
            if not has_both:
                warnings['missing_lods'] += 1
            status = "" if has_both else " ⚠ missing LOD"
            print(f"      └─ {child_id}")
            print(f"         LODs: {lods}{status}")

        shown += 1

    print(f"\n  SUMMARY: {len(buildings)} buildings")
    for w, count in warnings.items():
        if count > 0:
            print(f"    WARNING — {w}: {count}")

    return buildings
'''

Key Attributes

In [ ]:
def explore_attributes(cj):
    """Show 3DBAG attributes with missing-value tracking."""
    print(f"\n{'='*70}")
    print("3DBAG BUILDING ATTRIBUTES")
    print(f"{'='*70}")

    print(f"""
KEY ATTRIBUTES:
  b3_dak_type       → 'horizontal'/'slanted'/'multiple' (NOT gabled vs hipped!)
  b3_h_dak_min/max  → Min/max roof height (eave/ridge)
  b3_volume_lod13   → Volume from LOD1.3
  b3_volume_lod22   → Volume from LOD2.2
""")

    city_objects = cj.get('CityObjects', {})
    buildings = {k: v for k, v in city_objects.items()
                 if v.get('type') == 'Building'}

    if not buildings:
        print("  No Building objects found.")
        return

    # Track attribute availability
    tracked_attrs = ['b3_dak_type', 'b3_h_dak_min', 'b3_h_dak_max',
                     'b3_h_dak_50p', 'b3_h_dak_70p', 'b3_h_maaiveld',
                     'b3_volume_lod13', 'b3_volume_lod22',
                     'oorspronkelijk_bouwjaar']
    attr_present = Counter()
    attr_missing = Counter()
    dak_type_counts = Counter()

    for bld_id, bld in buildings.items():
        attrs = bld.get('attributes')
        if not attrs or not isinstance(attrs, dict):
            for key in tracked_attrs:
                attr_missing[key] += 1
            dak_type_counts['MISSING'] += 1
            continue

        for key in tracked_attrs:
            val = attrs.get(key)
            if val is not None and val != "":
                attr_present[key] += 1
            else:
                attr_missing[key] += 1

        dak = attrs.get('b3_dak_type')
        dak_type_counts[dak if dak else 'MISSING'] += 1

    # Availability report
    total = len(buildings)
    print(f"Attribute availability ({total} buildings):")
    print(f"  {'Attribute':<30} {'Present':<10} {'Missing':<10} {'Coverage'}")
    print(f"  {'-'*60}")
    for key in tracked_attrs:
        present = attr_present.get(key, 0)
        missing = attr_missing.get(key, 0)
        coverage = f"{present/total*100:.1f}%" if total > 0 else "N/A"
        print(f"  {key:<30} {present:<10} {missing:<10} {coverage}")

    print(f"\n  Roof type distribution:")
    for dak, count in dak_type_counts.most_common():
        print(f"    {dak:<15} {count:>6} ({count/total*100:.1f}%)")

    # Detail table
    max_show = min(15, total)
    print(f"\n  Detail (first {max_show}):")
    header = f"  {'Short ID':<30} {'Dak':<12} {'H_min':<8} {'H_max':<8} {'Vol13':<10} {'Vol22':<10} {'ΔVol'}"
    print(header)
    print(f"  {'-'*90}")

    shown = 0
    for bld_id, bld in buildings.items():
        if shown >= max_show:
            break

        attrs = bld.get('attributes') or {}
        dak = attrs.get('b3_dak_type', 'N/A')
        h_min = safe_float(attrs.get('b3_h_dak_min'))
        h_max = safe_float(attrs.get('b3_h_dak_max'))
        vol13 = safe_float(attrs.get('b3_volume_lod13'))
        vol22 = safe_float(attrs.get('b3_volume_lod22'))

        h_min_s = f"{h_min:.1f}" if h_min is not None else "N/A"
        h_max_s = f"{h_max:.1f}" if h_max is not None else "N/A"
        vol13_s = f"{vol13:.1f}" if vol13 is not None else "N/A"
        vol22_s = f"{vol22:.1f}" if vol22 is not None else "N/A"
        vol_d = f"{(vol13-vol22)/vol13*100:+.1f}%" if (vol13 and vol22 and vol13 > 0) else "N/A"

        short = bld_id[-28:] if len(bld_id) > 28 else bld_id
        print(f"  {short:<30} {dak:<12} {h_min_s:<8} {h_max_s:<8} {vol13_s:<10} {vol22_s:<10} {vol_d}")
        shown += 1

GEOMETRY HELPERS

In [8]:
def get_real_vertices(cj):
    """Convert all integer vertices to real-world coordinates."""
    transform = cj.get('transform', {})
    scale = transform.get('scale', [1, 1, 1])
    translate = transform.get('translate', [0, 0, 0])

    real_verts = []
    for v in cj.get('vertices', []):
        if isinstance(v, (list, tuple)) and len(v) >= 3:
            real_verts.append([
                v[0] * scale[0] + translate[0],
                v[1] * scale[1] + translate[1],
                v[2] * scale[2] + translate[2]
            ])
        else:
            real_verts.append([0.0, 0.0, 0.0])
    return real_verts

In [9]:
def get_shell_faces(boundaries, geom_type):
    """Get the list of faces (outer shell for Solids)."""
    try:
        if geom_type == 'Solid':
            return boundaries[0] if boundaries else []
        elif geom_type in ('MultiSurface', 'CompositeSurface'):
            return boundaries if isinstance(boundaries, list) else []
        else:
            return boundaries if isinstance(boundaries, list) else []
    except (TypeError, IndexError):
        return []

In [10]:
def get_semantic_values(semantics, geom_type):
    """Extract the flat list of semantic indices from semantics.values."""
    if not semantics:
        return [], []
    surfaces = semantics.get('surfaces', [])
    values_outer = semantics.get('values', [])

    if not values_outer:
        return surfaces, []

    # For Solid: values = [[0,1,2,...]]  (list per shell)
    # For MultiSurface: values = [0,1,2,...]
    if values_outer and isinstance(values_outer, list):
        if len(values_outer) > 0 and isinstance(values_outer[0], list):
            values = values_outer[0]  # Solid → take outer shell
        else:
            values = values_outer
    else:
        values = []

    return surfaces, values

In [ ]:
'''
def safe_vertex(real_verts, idx, total_verts):
    """Safely get a real vertex by index, return None if invalid."""
    if isinstance(idx, int) and 0 <= idx < total_verts:
        return real_verts[idx]
    return None
'''

In [12]:
def get_face_vertex_indices(face):
    """Extract vertex indices from a face (which has outer ring + optional holes)."""
    if not face:
        return []
    ring = face[0] if face and isinstance(face[0], list) else face
    return [idx for idx in ring if isinstance(idx, int)]


Geometry Deep Dive

In [13]:
def explore_geometry_detail(cj, max_buildings=5):
    """Deep dive into LOD1.3 vs LOD2.2 geometry."""
    print(f"\n{'='*70}")
    print(f"GEOMETRY DEEP DIVE (showing up to {max_buildings} buildings)")
    print(f"{'='*70}")

    real_verts = get_real_vertices(cj)
    total_verts = len(real_verts)
    city_objects = cj.get('CityObjects', {})

    parts = {k: v for k, v in city_objects.items()
             if v.get('type') == 'BuildingPart'}

    if not parts:
        print("  No BuildingPart objects found.")
        return

    shown = 0
    for part_id, part in parts.items():
        if shown >= max_buildings:
            remaining = len(parts) - shown
            if remaining > 0:
                print(f"\n  ... {remaining} more BuildingParts not shown")
            break

        parents = part.get('parents', [])
        parent_id = parents[0] if parents else None
        parent = city_objects.get(parent_id, {}) if parent_id else {}
        dak = (parent.get('attributes') or {}).get('b3_dak_type', 'N/A')

        print(f"\n{'─'*70}")
        print(f"BuildingPart: {part_id}")
        print(f"Roof type (b3_dak_type): {dak}")
        if not parents:
            print("WARNING: No parent Building")
        print(f"{'─'*70}")

        geom_list = part.get('geometry', [])
        if not geom_list:
            print("  WARNING: No geometry found")
            shown += 1
            continue

        for geom in geom_list:
            lod = geom.get('lod', '?')
            geom_type = geom.get('type', 'unknown')
            boundaries = geom.get('boundaries', [])

            print(f"\n  LOD {lod} ({geom_type}):")

            if not boundaries:
                print("    WARNING: Empty boundaries")
                continue

            shell = get_shell_faces(boundaries, geom_type)
            print(f"    Faces: {len(shell)}")

            # Collect all valid vertex indices
            all_indices = set()
            for face in shell:
                for idx in get_face_vertex_indices(face):
                    if 0 <= idx < total_verts:
                        all_indices.add(idx)

            invalid_count = sum(1 for face in shell
                               for idx in get_face_vertex_indices(face)
                               if not (0 <= idx < total_verts))
            if invalid_count:
                print(f"    WARNING: {invalid_count} invalid vertex indices")

            print(f"    Unique vertices: {len(all_indices)}")

            # Semantics
            semantics = geom.get('semantics', {})
            if semantics:
                surfaces, values = get_semantic_values(semantics, geom_type)
                type_counts = Counter()
                for val in values:
                    if val is not None and isinstance(val, int) and 0 <= val < len(surfaces):
                        type_counts[surfaces[val].get('type', '?')] += 1
                    elif val is None:
                        type_counts['(null)'] += 1
                    else:
                        type_counts['(invalid ref)'] += 1

                print(f"    Semantics: {dict(type_counts)}")

                for i, surf in enumerate(surfaces):
                    if surf.get('type') == 'RoofSurface':
                        slope = surf.get('b3_hellingshoek')
                        if slope is not None:
                            print(f"      RoofSurface[{i}] slope: {slope}°")
            elif lod == '2.2':
                print("    WARNING: LOD2.2 has no semantics!")

            # Height analysis
            if all_indices:
                z_vals = [real_verts[i][2] for i in all_indices]
                print(f"\n    Height: min={min(z_vals):.3f}, max={max(z_vals):.3f}, "
                      f"range={max(z_vals)-min(z_vals):.3f} m")

                if lod == '2.2':
                    unique_z = sorted(set(round(z, 2) for z in z_vals))
                    print(f"    Z-levels: {unique_z[:10]}")
                    if len(unique_z) >= 3:
                        print(f"    → Ground≈{unique_z[0]:.2f}, Eave≈{unique_z[1]:.2f}, "
                              f"Ridge≈{unique_z[-1]:.2f}")

            # Vertex list (abbreviated)
            sorted_idx = sorted(all_indices)
            max_v = min(10, len(sorted_idx))
            print(f"\n    Vertices (first {max_v}):")
            for idx in sorted_idx[:max_v]:
                v = real_verts[idx]
                print(f"      [{idx:>5}] x={v[0]:>12.3f} y={v[1]:>12.3f} z={v[2]:>8.3f}")
            if len(sorted_idx) > max_v:
                print(f"      ... +{len(sorted_idx)-max_v} more")

            # Face details (abbreviated)
            max_f = min(10, len(shell))
            print(f"\n    Faces (first {max_f}):")
            for fi, face in enumerate(shell[:max_f]):
                ring = get_face_vertex_indices(face)
                valid = [i for i in ring if 0 <= i < total_verts]
                z_range = (f"z=[{min(real_verts[i][2] for i in valid):.2f}, "
                           f"{max(real_verts[i][2] for i in valid):.2f}]") if valid else "z=[N/A]"

                sem_label = ""
                if semantics and fi < len(values):
                    si = values[fi]
                    if si is not None and isinstance(si, int) and 0 <= si < len(surfaces):
                        sem_label = f" [{surfaces[si].get('type', '?')}]"

                verts_str = str(ring[:8])
                if len(ring) > 8:
                    verts_str += f"...(+{len(ring)-8})"
                print(f"      Face {fi}{sem_label}: {verts_str} {z_range}")

            if len(shell) > max_f:
                print(f"      ... +{len(shell)-max_f} more faces")

        shown += 1

Roof Type Analysis

In [14]:
def analyze_roof_types(cj, max_show=15):
    """Determine roof type from LOD2.2 RoofSurface count."""
    print(f"\n{'='*70}")
    print("ROOF TYPE ANALYSIS FROM LOD2.2")
    print(f"{'='*70}")

    print(f"""
Determining roof type by counting RoofSurface faces in LOD2.2:
  1 RoofSurface  → FLAT
  2 RoofSurfaces → GABLED
  4 RoofSurfaces → HIPPED
  Other          → COMPLEX (out of scope)
""")

    real_verts = get_real_vertices(cj)
    total_verts = len(real_verts)
    city_objects = cj.get('CityObjects', {})

    derived_counts = Counter()
    issues = {'no_lod22': 0, 'no_semantics': 0, 'mismatch': 0}
    shown = 0

    for obj_id, obj in city_objects.items():
        if obj.get('type') != 'BuildingPart':
            continue

        parents = obj.get('parents', [])
        parent_id = parents[0] if parents else None
        parent = city_objects.get(parent_id, {}) if parent_id else {}
        b3_dak = (parent.get('attributes') or {}).get('b3_dak_type', 'N/A')

        # Find LOD2.2
        lod22_geom = None
        for geom in obj.get('geometry', []):
            if geom.get('lod') == '2.2':
                lod22_geom = geom
                break

        if not lod22_geom:
            issues['no_lod22'] += 1
            continue

        semantics = lod22_geom.get('semantics')
        if not semantics:
            issues['no_semantics'] += 1
            continue

        surfaces, values = get_semantic_values(semantics, lod22_geom.get('type', 'Solid'))
        boundaries = lod22_geom.get('boundaries', [])
        shell = get_shell_faces(boundaries, lod22_geom.get('type', 'Solid'))

        # Count roof faces and collect vertices
        roof_count = 0
        roof_verts = set()
        for fi, val in enumerate(values):
            if val is None or not isinstance(val, int) or val >= len(surfaces):
                continue
            if surfaces[val].get('type') == 'RoofSurface':
                roof_count += 1
                if fi < len(shell):
                    for idx in get_face_vertex_indices(shell[fi]):
                        if 0 <= idx < total_verts:
                            roof_verts.add(idx)

        # Derive type
        if roof_count == 0:
            derived = "UNKNOWN (0 roof faces)"
        elif roof_count == 1:
            derived = "FLAT"
        elif roof_count == 2:
            derived = "GABLED"
        elif roof_count == 4:
            derived = "HIPPED"
        else:
            derived = f"COMPLEX ({roof_count} faces)"

        derived_counts[derived] += 1

        # Check mismatch
        if (b3_dak == 'horizontal' and derived != 'FLAT') or \
           (b3_dak == 'slanted' and derived == 'FLAT'):
            issues['mismatch'] += 1

        # Show details
        if shown < max_show:
            print(f"\n  ...{obj_id[-35:]}")
            print(f"    b3_dak_type: {b3_dak} → Derived: {derived} ({roof_count} roof faces)")

            if roof_verts:
                roof_z = [real_verts[i][2] for i in roof_verts]
                eave = min(roof_z)
                ridge = max(roof_z)
                print(f"    Eave: {eave:.3f} m, Ridge: {ridge:.3f} m, Rise: {ridge-eave:.3f} m")

                if derived in ("GABLED", "HIPPED"):
                    ridge_v = [i for i in roof_verts
                               if abs(real_verts[i][2] - ridge) < 0.05]
                    if ridge_v:
                        print(f"    Ridge vertices ({len(ridge_v)}):")
                        for rv in ridge_v[:4]:
                            v = real_verts[rv]
                            print(f"      [{rv}] ({v[0]:.3f}, {v[1]:.3f}, {v[2]:.3f})")

            shown += 1

    # Summary
    print(f"\n{'─'*70}")
    print(f"SUMMARY:")
    for dtype, count in derived_counts.most_common():
        print(f"  {dtype:<35} {count}")
    for issue, count in issues.items():
        if count > 0:
            print(f"  Issue — {issue}: {count}")

Feature Extraction Preview

In [ ]:
def extract_lod13_features(cj, max_show=10):
    """Preview LOD1.3 feature extraction."""
    print(f"\n{'='*70}")
    print("LOD1.3 FEATURE EXTRACTION PREVIEW")
    print(f"{'='*70}")

    print(f"""
Features from LOD1.3 for ML:
  1. Footprint area (m²)       5. Compactness
  2. Perimeter (m)             6. Building height (m)
  3. Footprint vertices        7. Longest edge orientation
  4. Aspect ratio              8. Volume difference %
""")

    real_verts = get_real_vertices(cj)
    total_verts = len(real_verts)
    city_objects = cj.get('CityObjects', {})

    shown = 0
    errors = 0

    for obj_id, obj in city_objects.items():
        if obj.get('type') != 'BuildingPart':
            continue
        if shown >= max_show:
            break

        parents = obj.get('parents', [])
        parent_id = parents[0] if parents else None
        parent = city_objects.get(parent_id, {}) if parent_id else {}
        parent_attrs = parent.get('attributes') or {}

        # Find LOD1.3
        lod13 = None
        for geom in obj.get('geometry', []):
            if geom.get('lod') == '1.3':
                lod13 = geom
                break

        if not lod13:
            continue

        geom_type = lod13.get('type', 'Solid')
        shell = get_shell_faces(lod13.get('boundaries', []), geom_type)
        if not shell:
            errors += 1
            continue

        # Find ground face (lowest avg z)
        ground_face = None
        min_avg_z = float('inf')

        for face in shell:
            ring = get_face_vertex_indices(face)
            valid = [i for i in ring if 0 <= i < total_verts]
            if not valid:
                continue
            avg_z = sum(real_verts[i][2] for i in valid) / len(valid)
            if avg_z < min_avg_z:
                min_avg_z = avg_z
                ground_face = valid

        if not ground_face or len(ground_face) < 3:
            errors += 1
            continue

        # 2D footprint
        fp = [(real_verts[i][0], real_verts[i][1]) for i in ground_face]
        n = len(fp)

        # Area (shoelace)
        area = 0.0
        for i in range(n):
            j = (i + 1) % n
            area += fp[i][0] * fp[j][1] - fp[j][0] * fp[i][1]
        area = abs(area) / 2.0

        if area < 0.1:
            errors += 1
            continue

        # Perimeter + edge lengths
        perimeter = 0.0
        edges = []
        for i in range(n):
            j = (i + 1) % n
            dx = fp[j][0] - fp[i][0]
            dy = fp[j][1] - fp[i][1]
            length = math.sqrt(dx*dx + dy*dy)
            perimeter += length
            edges.append((length, i, j))

        compactness = (4 * math.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

        # Height
        all_idx = set()
        for face in shell:
            for i in get_face_vertex_indices(face):
                if 0 <= i < total_verts:
                    all_idx.add(i)
        z_vals = [real_verts[i][2] for i in all_idx]
        height = (max(z_vals) - min(z_vals)) if z_vals else 0

        # Longest edge + orientation
        if edges:
            longest = max(edges, key=lambda x: x[0])
            ll = longest[0]
            li, lj = longest[1], longest[2]
            dx = fp[lj][0] - fp[li][0]
            dy = fp[lj][1] - fp[li][1]
            orient = math.degrees(math.atan2(dy, dx)) % 180
        else:
            ll, orient = 0, 0

        aspect = (ll / (area / ll)) if ll > 0 else 1.0

        # Volume from attributes
        vol13 = safe_float(parent_attrs.get('b3_volume_lod13'))
        vol22 = safe_float(parent_attrs.get('b3_volume_lod22'))
        vol_red = f"{(vol13-vol22)/vol13*100:.1f}%" if (vol13 and vol22 and vol13 > 0) else "N/A"

        print(f"\n  ...{obj_id[-35:]}")
        print(f"    Vertices: {n}  |  Area: {area:.2f} m²  |  Perimeter: {perimeter:.2f} m")
        print(f"    Height: {height:.2f} m  |  Compactness: {compactness:.4f}  |  Aspect: {aspect:.2f}")
        print(f"    Longest edge: {ll:.2f} m @ {orient:.1f}°")
        print(f"    Volume LOD1.3: {vol13 if vol13 else 'N/A'} | LOD2.2: {vol22 if vol22 else 'N/A'} | Δ: {vol_red}")
        shown += 1

    if errors:
        print(f"\n {errors} building(s) skipped (missing/invalid geometry)")

LOD Comparison

In [16]:
def compare_lod13_vs_lod22(cj, max_show=15):
    """Compare LOD1.3 and LOD2.2 side by side."""
    print(f"\n{'='*70}")
    print("LOD1.3 vs LOD2.2 COMPARISON")
    print(f"{'='*70}")

    print(f"""
┌───────────────────┬──────────────────┬─────────────────────────┐
│ Property          │ LOD1.3           │ LOD2.2                  │
├───────────────────┼──────────────────┼─────────────────────────┤
│ Roof              │ Flat top         │ Actual shape            │
│ Semantics         │ NO               │ YES (Roof/Wall/Ground)  │
│ Vertices          │ Fewer (box)      │ More (+ roof verts)     │
│ Volume            │ Overestimates    │ Accurate                │
└───────────────────┴──────────────────┴─────────────────────────┘
""")

    city_objects = cj.get('CityObjects', {})
    shown = 0

    for obj_id, obj in city_objects.items():
        if obj.get('type') != 'BuildingPart' or shown >= max_show:
            continue

        parents = obj.get('parents', [])
        parent_id = parents[0] if parents else None
        parent = city_objects.get(parent_id, {}) if parent_id else {}
        dak = (parent.get('attributes') or {}).get('b3_dak_type', 'N/A')

        lod13 = lod22 = None
        for geom in obj.get('geometry', []):
            if geom.get('lod') == '1.3':
                lod13 = geom
            elif geom.get('lod') == '2.2':
                lod22 = geom

        if not lod13 or not lod22:
            continue

        def count_geom(g):
            gt = g.get('type', 'Solid')
            shell = get_shell_faces(g.get('boundaries', []), gt)
            verts = set()
            for face in shell:
                for idx in get_face_vertex_indices(face):
                    verts.add(idx)
            return len(shell), len(verts)

        f13, v13 = count_geom(lod13)
        f22, v22 = count_geom(lod22)

        # Semantics
        semantics = lod22.get('semantics')
        surf_types = {}
        if semantics:
            surfaces, values = get_semantic_values(semantics, lod22.get('type', 'Solid'))
            for val in values:
                if val is not None and isinstance(val, int) and 0 <= val < len(surfaces):
                    st = surfaces[val].get('type', '?')
                    surf_types[st] = surf_types.get(st, 0) + 1

        sem_str = str(surf_types) if surf_types else "NO semantics"
        print(f"  ...{obj_id[-35:]} (dak: {dak})")
        print(f"    LOD1.3: {f13} faces, {v13} verts")
        print(f"    LOD2.2: {f22} faces, {v22} verts, {sem_str}")
        print(f"    Δ: +{v22-v13} verts, +{f22-f13} faces\n")
        shown += 1

Data Quality Report

In [17]:
def data_quality_report(cj):
    """Summarize data quality for pipeline planning."""
    print(f"\n{'='*70}")
    print("DATA QUALITY REPORT")
    print(f"{'='*70}")

    city_objects = cj.get('CityObjects', {})
    total_verts = len(cj.get('vertices', []))

    buildings = {k: v for k, v in city_objects.items()
                 if v.get('type') == 'Building'}
    parts = {k: v for k, v in city_objects.items()
             if v.get('type') == 'BuildingPart'}

    issues = Counter()
    usable = 0

    for bld_id, bld in buildings.items():
        attrs = bld.get('attributes') or {}
        children = bld.get('children', [])

        if not attrs:
            issues['no_attributes'] += 1
        if not attrs.get('b3_dak_type'):
            issues['no_dak_type'] += 1
        if not children:
            issues['no_children'] += 1
            continue
        if len(children) > 1:
            issues['multiple_parts'] += 1

        child = city_objects.get(children[0])
        if not child:
            issues['missing_child_ref'] += 1
            continue

        geom_list = child.get('geometry', [])
        if not geom_list:
            issues['no_geometry'] += 1
            continue

        has13 = any(g.get('lod') == '1.3' for g in geom_list)
        has22 = any(g.get('lod') == '2.2' for g in geom_list)

        if not has13:
            issues['no_lod13'] += 1
        if not has22:
            issues['no_lod22'] += 1

        if has22:
            lod22 = next(g for g in geom_list if g.get('lod') == '2.2')
            if not lod22.get('semantics'):
                issues['no_semantics'] += 1

        if has13 and has22 and len(children) == 1:
            usable += 1

    total = len(buildings)
    pct = f"{usable/total*100:.1f}%" if total else "N/A"

    print(f"\n  Total Buildings:    {total}")
    print(f"  Total Parts:        {len(parts)}")
    print(f"  Total Vertices:     {total_verts}")
    print(f"\n  USABLE (1 part + both LODs): {usable} ({pct})")
    print(f"\n  Issues:")
    for issue, count in issues.most_common():
        print(f"    {issue:<25} {count}")

In [18]:
filepath = r"C:\Users\HP\Desktop\Bachelor's-Thesis\8-600-584.city.json"
cj = load_cityjson(filepath)


Loading: C:\Users\HP\Desktop\Bachelor's-Thesis\8-600-584.city.json

CityJSON version: 2.0
Top-level keys: ['CityObjects', 'metadata', 'transform', 'type', 'version', 'vertices']

Transform:
  Scale:     [0.001, 0.001, 0.001]
  Translate: [164589.09375, 453885.75, 13.779500961303711]
  → Real coordinate = (integer_vertex * scale) + translate

Total shared vertices: 90649
Total CityObjects: 3671
Reference system: https://www.opengis.net/def/crs/EPSG/0/7415


In [ ]:
show_transform_example(cj)


VERTEX TRANSFORMATION

CityJSON stores vertices as INTEGERS to save space.
Real-world coords: real = integer * scale + translate

Scale:     [0.001, 0.001, 0.001]
Translate: [164589.09375, 453885.75, 13.779500961303711]

Index    Integer (x,y,z)           Real-world (x,y,z) [meters]
----------------------------------------------------------------------
0        [-824044, -843078, -13779] (163765.050, 453042.672, 0.001)
1        [-812677, -841835, -13779] (163776.417, 453043.915, 0.001)
2        [-815342, -816853, -13779] (163773.752, 453068.897, 0.001)
3        [-826827, -817978, -13779] (163762.267, 453067.772, 0.001)
4        [-826827, -817978, -7574] (163762.267, 453067.772, 6.206)
5        [-815342, -816853, -7574] (163773.752, 453068.897, 6.206)


([0.001, 0.001, 0.001], [164589.09375, 453885.75, 13.779500961303711])

In [27]:
# explore_city_objects(cj)

In [21]:
explore_attributes(cj)


3DBAG BUILDING ATTRIBUTES

KEY ATTRIBUTES:
  b3_dak_type       → 'horizontal'/'slanted'/'multiple' (NOT gabled vs hipped!)
  b3_h_dak_min/max  → Min/max roof height (eave/ridge)
  b3_volume_lod13   → Volume from LOD1.3
  b3_volume_lod22   → Volume from LOD2.2

Attribute availability (1834 buildings):
  Attribute                      Present    Missing    Coverage
  ------------------------------------------------------------
  b3_dak_type                    1834       0          100.0%
  b3_h_dak_min                   1831       3          99.8%
  b3_h_dak_max                   1831       3          99.8%
  b3_h_dak_50p                   1831       3          99.8%
  b3_h_dak_70p                   1834       0          100.0%
  b3_h_maaiveld                  1834       0          100.0%
  b3_volume_lod13                1834       0          100.0%
  b3_volume_lod22                1834       0          100.0%
  oorspronkelijk_bouwjaar        0          1834       0.0%

  Roof type dist

In [22]:
explore_geometry_detail(cj, max_buildings=5)


GEOMETRY DEEP DIVE (showing up to 5 buildings)

──────────────────────────────────────────────────────────────────────
BuildingPart: NL.IMBAG.Pand.0279100000003466-0
Roof type (b3_dak_type): slanted
──────────────────────────────────────────────────────────────────────

  LOD 1.2 (Solid):
    Faces: 6
    Unique vertices: 8
    Semantics: {'GroundSurface': 1, 'WallSurface': 4, 'RoofSurface': 1}

    Height: min=6.638, max=12.022, range=5.384 m

    Vertices (first 8):
      [ 4874] x=  164161.284 y=  454589.058 z=   6.638
      [ 4875] x=  164153.213 y=  454583.263 z=   6.638
      [ 4876] x=  164142.888 y=  454597.649 z=   6.638
      [ 4877] x=  164150.961 y=  454603.443 z=   6.638
      [ 4878] x=  164161.284 y=  454589.058 z=  12.022
      [ 4879] x=  164150.961 y=  454603.443 z=  12.022
      [ 4880] x=  164142.888 y=  454597.649 z=  12.022
      [ 4881] x=  164153.213 y=  454583.263 z=  12.022

    Faces (first 6):
      Face 0 [GroundSurface]: [4874, 4875, 4876, 4877] z=[6.64, 

In [23]:
analyze_roof_types(cj, max_show=15)


ROOF TYPE ANALYSIS FROM LOD2.2

Determining roof type by counting RoofSurface faces in LOD2.2:
  1 RoofSurface  → FLAT
  2 RoofSurfaces → GABLED
  4 RoofSurfaces → HIPPED
  Other          → COMPLEX (out of scope)


  ...NL.IMBAG.Pand.0279100000003466-0
    b3_dak_type: slanted → Derived: GABLED (2 roof faces)
    Eave: 8.390 m, Ridge: 13.682 m, Rise: 5.292 m
    Ridge vertices (1):
      [4889] (164157.266, 454586.173, 13.682)

  ...NL.IMBAG.Pand.0279100000003467-0
    b3_dak_type: slanted → Derived: COMPLEX (5 faces) (5 roof faces)
    Eave: 8.101 m, Ridge: 12.073 m, Rise: 3.972 m

  ...NL.IMBAG.Pand.0279100000003681-0
    b3_dak_type: slanted → Derived: COMPLEX (5 faces) (5 roof faces)
    Eave: 8.196 m, Ridge: 14.422 m, Rise: 6.226 m

  ...NL.IMBAG.Pand.0279100000006195-0
    b3_dak_type: slanted → Derived: GABLED (2 roof faces)
    Eave: 9.675 m, Ridge: 12.966 m, Rise: 3.291 m
    Ridge vertices (2):
      [4868] (164113.640, 454593.586, 12.950)
      [4863] (164136.204, 454608.53

In [24]:
extract_lod13_features(cj, max_show=10)


LOD1.3 FEATURE EXTRACTION PREVIEW

Features from LOD1.3 for ML:
  1. Footprint area (m²)       5. Compactness
  2. Perimeter (m)             6. Building height (m)
  3. Footprint vertices        7. Longest edge orientation
  4. Aspect ratio              8. Volume difference %


  ...NL.IMBAG.Pand.0279100000003466-0
    Vertices: 4  |  Area: 175.94 m²  |  Perimeter: 55.29 m
    Height: 5.38 m  |  Compactness: 0.7233  |  Aspect: 1.78
    Longest edge: 17.71 m @ 125.7°
    Volume LOD1.3: N/A | LOD2.2: N/A | Δ: N/A

  ...NL.IMBAG.Pand.0279100000003467-0
    Vertices: 22  |  Area: 153.93 m²  |  Perimeter: 68.33 m
    Height: 4.07 m  |  Compactness: 0.4143  |  Aspect: 0.71
    Longest edge: 10.42 m @ 114.5°
    Volume LOD1.3: N/A | LOD2.2: N/A | Δ: N/A

  ...NL.IMBAG.Pand.0279100000003681-0
    Vertices: 4  |  Area: 260.07 m²  |  Perimeter: 66.30 m
    Height: 6.19 m  |  Compactness: 0.7435  |  Aspect: 1.64
    Longest edge: 20.63 m @ 119.1°
    Volume LOD1.3: N/A | LOD2.2: N/A | Δ: N/A

  

In [25]:
compare_lod13_vs_lod22(cj, max_show=15)


LOD1.3 vs LOD2.2 COMPARISON

┌───────────────────┬──────────────────┬─────────────────────────┐
│ Property          │ LOD1.3           │ LOD2.2                  │
├───────────────────┼──────────────────┼─────────────────────────┤
│ Roof              │ Flat top         │ Actual shape            │
│ Semantics         │ NO               │ YES (Roof/Wall/Ground)  │
│ Vertices          │ Fewer (box)      │ More (+ roof verts)     │
│ Volume            │ Overestimates    │ Accurate                │
└───────────────────┴──────────────────┴─────────────────────────┘

  ...NL.IMBAG.Pand.0279100000003466-0 (dak: slanted)
    LOD1.3: 6 faces, 8 verts
    LOD2.2: 9 faces, 12 verts, {'GroundSurface': 1, 'WallSurface': 6, 'RoofSurface': 2}
    Δ: +4 verts, +3 faces

  ...NL.IMBAG.Pand.0279100000003467-0 (dak: slanted)
    LOD1.3: 24 faces, 44 verts
    LOD2.2: 39 faces, 64 verts, {'GroundSurface': 1, 'WallSurface': 33, 'RoofSurface': 5}
    Δ: +20 verts, +15 faces

  ...NL.IMBAG.Pand.02791000000036

In [26]:
data_quality_report(cj)


DATA QUALITY REPORT

  Total Buildings:    1834
  Total Parts:        1837
  Total Vertices:     90649

  USABLE (1 part + both LODs): 1831 (99.8%)

  Issues:
    multiple_parts            3
